In [1]:
import numpy as np
import xarray as xr

In [2]:
from data import ds_pressure, ds_single

In [3]:
from dataclasses import dataclass


@dataclass
class RangeBounds:
    min: float
    max: float
    min_abs: float

In [4]:
def compute_range_bounds(da: xr.DataArray) -> RangeBounds:
    xs = da.values.flatten()

    xs_no_zeros = xs[xs != 0.0]

    return RangeBounds(
        min=np.nanmin(xs),
        max=np.nanmax(xs),
        min_abs=np.nanmin(np.abs(xs_no_zeros)) if len(xs_no_zeros) > 0 else 0.0,
    )

In [5]:
from tqdm import tqdm

In [6]:
with open("range-single.csv", "w") as f:
    print("name,level,min,max,min_abs", flush=True, file=f)
    for key in tqdm(sorted(ds_single.mean)):
        bounds = compute_range_bounds(ds_single.mean[key])
        print(
            f"{key},single,{bounds.min},{bounds.max},{bounds.min_abs}",
            flush=True,
            file=f,
        )

100%|███████████████████████████████████████████████████████████| 226/226 [04:42<00:00,  1.25s/it]


In [7]:
with open("range-pressure.csv", "w") as f:
    print("name,level,min,max,min_abs", flush=True, file=f)
    for key in tqdm(sorted(ds_pressure.mean)):
        for pressure in sorted(ds_pressure.mean[key].pressure_level.values):
            bounds = compute_range_bounds(
                ds_pressure.mean[key].sel(pressure_level=pressure)
            )
            print(
                f"{key},{pressure}hPa,{bounds.min},{bounds.max},{bounds.min_abs}",
                flush=True,
                file=f,
            )

100%|█████████████████████████████████████████████████████████████| 16/16 [01:25<00:00,  5.34s/it]
